In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pyvisa as visa
import time

kHz = 1e3
MHz = 1e6
GHz = 1e9

In [2]:
def setupXaxis(f_center, f_span, n_points, active_channel=1):
    
    kna.write(f":SENS{active_channel}:FREQ:CENT {f_center}")
    time.sleep(0.2)
    kna.write(f":SENS{active_channel}:FREQ:SPAN {f_span}")
    time.sleep(0.2)
    kna.write(f":SENS{active_channel}:SWE:POIN {n_points}")
    time.sleep(0.2)
#     kna.write(f":SENS{active_channel}:SWE:TYPE LIN")
#     time.sleep(1)

    x_array = np.linspace(f_center - f_span/2, f_center + f_span/2, n_points)
    
    return x_array


def setupMeasurement(s_ij = "S21", meas_format = "COMP", active_channel=1):
    
    # Supply all arguments in correct units
    
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:PAR {s_ij};")
    time.sleep(0.5)
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:FORM {meas_format};")
    time.sleep(0.5)


def setupAveraging(n_avgs, IF_bw):
    
    kna.write(f":SENS{active_channel}:BWID {IF_bw}")
    time.sleep(0.5)
    kna.write(f":SENS{active_channel}:AVER ON")
    time.sleep(0.5)
    kna.write(f":SENS{active_channel}:AVER:COUN {n_avgs}")
    time.sleep(0.5)
    
def setPower_getRealImag(power_dBm, wait_time, active_channel=1):
    
    kna.write(f":SOUR{active_channel}:POW {power_dBm}")
    time.sleep(wait_time)
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:DATA:FDAT?")
    ydata_str = kna.read()
    ydata_temp = ydata_str.split(",")
    y_data = np.array([float(d) for d in ydata_temp])
    y_data = y_data.reshape(n_points, 2)
    y_data = y_data.transpose()
    
    real, imag = y_data
    
    return real, imag

def save_data(data, fname):
    
    path = r"D:\Experiments\2024-02-13 RingV1 wo qubits and 7 paramps\Low Temperature Response\Ring3_Input5Output29\\"
    data = np.transpose(data)
    
    np.savetxt(path + fname, data)
    
def getRealImag(active_channel=1):
    
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:DATA:FDAT?")
    ydata_str = kna.read()
    ydata_temp = ydata_str.split(",")
    y_data = np.array([float(d) for d in ydata_temp])
    y_data = y_data.reshape(n_points, 2)
    y_data = y_data.transpose()
    
    real, imag = y_data
    
    return real, imag

In [3]:
ip = "TCPIP::192.168.0.27"
# ip = "TCPIP0::DESKTOP-S092ETA::hislip_PXI10_CHASSIS1_SLOT1_INDEX0::INSTR"
rm = visa.ResourceManager()
kna = rm.open_resource(ip)

In [4]:
kna

<'TCPIPInstrument'('TCPIP0::192.168.0.27::inst0::INSTR')>

In [5]:
kna.timeout =  10000
#kna.write("*IDN?")
#kna.write("SYST:PRES; *OPC?")

In [15]:
active_channel = 1
n_traces = 1
active_trace = 1
s_ij = "S21"

IF_bw = 1 * kHz
n_points = 2001

f_center = 7.699 * GHz
f_span = 80 * MHz

n_avgs = 50

In [16]:
x_array = setupXaxis(f_center, f_span, n_points)
time.sleep(2)
setupMeasurement(s_ij)
setupAveraging(n_avgs, IF_bw)

In [17]:
kna.write(f":SENS{active_channel}:SWE:TIME?")
sweep_time = float(kna.read())

In [18]:
power_min, power_max, power_step = -40, 10, 5
power_list = np.arange(power_min, power_max + power_step/2 , power_step)

for power_dBm in power_list:
    
    if power_dBm < -25:
        n_avgs = 50
        kna.write(f":SENS{active_channel}:AVER:COUN {n_avgs}")
        time.sleep(1)
    
    elif power_dBm < -15:
        n_avgs = 30
        kna.write(f":SENS{active_channel}:AVER:COUN {n_avgs}")
        time.sleep(1)
    
    else:
        n_avgs = 10
        kna.write(f":SENS{active_channel}:AVER:COUN {n_avgs}")
        time.sleep(1)
    #print(power_dBm)
    real, imag = setPower_getRealImag(power_dBm, sweep_time*n_avgs*1.2, active_channel=1)
    #time.sleep(sweep_time*n_avgs*1.2)
    
    save_data([x_array, real, imag], f"PowerSweep/Real_{power_dBm}_dBm.txt")

In [101]:
float(sweep_time)

1.442461

In [145]:
 real1, imag1 = getRealImag(active_channel=1)
 save_data([x_array, real1, imag1], f"data_-100_dBm.txt")
#this is going to just save the data from VNA, kept in polar chart. In the folder you've set in save_data function

In [ ]:
rm.close()